# Make a list of samples

In [7]:
from pathlib import Path

# Set global variables
work_dir = Path("/scratch/johansaenz/bled")
conda_path = "/DKHF/users/johansaenz/miniforge3/bin/conda"
db_path = Path("/scratch/johansaenz/DB")

# List samples from selected directory
input_dir = work_dir  / "rawdata"

samples = []

for read1 in sorted(input_dir.glob("*_1.fq.gz")):
    sample = read1.name.removesuffix("_1.fq.gz")
    read2 = input_dir / f"{sample}_2.fq.gz"

    if read2.is_file():
        samples.append(sample)
    else:
        print(f"Skipping {sample}: missing {read2.name}")

print(f"Found {len(samples)} paired samples:")
print(samples)

Found 4 paired samples:
['W000_E_RP', 'W000_T2_RP', 'W001_T2_RP', 'W002_T2_RP']


# Remove Illumina adapters

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor

# Set number of samples in parallel and threads 
max_samples = 2
cores_per_sample = 8
tool_name = "trimgalore"

# Set directories
input_dir = work_dir / "rawdata"
output_dir = work_dir / "trimgalore"
log_dir = work_dir / "log" / tool_name

output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

# Set command
def run_sample(sample):
    command = [
        conda_path, "run", "--no-capture-output",
        "-n", "trimgalore",
        "trim_galore",
        "--cores", str(cores_per_sample),
        "--paired",
        str(input_dir / f"{sample}_1.fq.gz"),
        str(input_dir / f"{sample}_2.fq.gz"),
        "--output_dir", str(output_dir),
    ]

    print(f"Starting: {sample}", flush=True)

    log_file = log_dir / f"{sample}.{tool_name}.log"

    with open(log_file, "w") as log:
        result = subprocess.run(
            command, stdout=log, stderr=subprocess.STDOUT
        )

    status = "Finished" if result.returncode == 0 else f"FAILED — check {log_file}"
    print(f"{sample}: {status}", flush=True)

with ThreadPoolExecutor(max_workers=max_samples) as executor:
    list(executor.map(run_sample, samples))

# Remove amplification adapters

In [9]:
import subprocess
from concurrent.futures import ThreadPoolExecutor

# Set number of samples in parallel and threads per sample 
max_samples = 2
cores_per_sample = 8
tool_name = "cutadapt"

# Script variables
input_dir = work_dir / "trimgalore"
output_dir = work_dir / "cutadapt"
log_dir = work_dir / "log" / tool_name

output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)


def run_sample(sample):
    command = [
        conda_path, "run", "--no-capture-output",
        "-n", "cutadapt",
        "cutadapt",
        "--cores", str(cores_per_sample),
        "-O", "5",
        "-g", "GTTTCCCAGTCACGATA",
        "-G", "GTTTCCCAGTCACGATA",
        "-a", "TATCGTGACTGGGAAAC",
        "-A", "TATCGTGACTGGGAAAC",
        "-o", str(output_dir / f"{sample}_1_cut.fq.gz"),
        "-p", str(output_dir / f"{sample}_2_cut.fq.gz"),
        str(input_dir / f"{sample}_1_val_1.fq.gz"),
        str(input_dir / f"{sample}_2_val_2.fq.gz")
    ]

    print(f"Starting: {sample}", flush=True)

    log_file = log_dir / f"{sample}.{tool_name}.log"

    with open(log_file, "w") as log:
        result = subprocess.run(
            command, stdout=log, stderr=subprocess.STDOUT
        )

    status = "Finished" if result.returncode == 0 else f"FAILED — check {log_file}"
    print(f"{sample}: {status}", flush=True)

with ThreadPoolExecutor(max_workers=max_samples) as executor:
    list(executor.map(run_sample, samples))

Starting: W000_E_RP
Starting: W000_T2_RP
W000_T2_RP: Finished
Starting: W001_T2_RP
W000_E_RP: Finished
Starting: W002_T2_RP
W001_T2_RP: Finished
W002_T2_RP: Finished


# Assemble contigs

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor

max_samples = 2
cores_per_sample = 10
memory_per_sample = 32000000000
tool_name = "megahit"


input_dir = work_dir / "cutadapt"
output_dir = work_dir / "megahit"
log_dir = work_dir / "log" / tool_name

output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

def run_sample(sample):
    command = [
        "conda", "run", "--no-capture-output",
        "-n", "megahit",
        "megahit",
        "-1", str(input_dir / f"{sample}_1_cut.fq.gz"),
        "-2", str(input_dir / f"{sample}_2_cut.fq.gz"),
        "-o", str(output_dir /f"{sample}"),
        "--min-contig-len", "3000",
        "--presets", "meta-sensitive",
        "-t", str(cores_per_sample),
        "-m", str(memory_per_sample),
    ]

    print(f"Starting: {sample}", flush=True)

    log_file = log_dir / f"{sample}.{tool_name}.log"

    with open(log_file, "w") as log:
        result = subprocess.run(
            command, stdout=log, stderr=subprocess.STDOUT
        )

    status = "Finished" if result.returncode == 0 else f"FAILED — check {log_file}"
    print(f"{sample}: {status}", flush=True)

with ThreadPoolExecutor(max_workers=max_samples) as executor:
    list(executor.map(run_sample, samples))

Starting: W001_T2_RP
Starting: W002_T2_RP


# Viral contigs identification

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor

max_samples = 2
cores_per_sample = 20
tool_name = "genomad"

input_dir = work_dir / "megahit"
output_dir = work_dir / "genomad"
log_dir = work_dir / "log" / tool_name
database = db_path / "genomad/genomad_db"

output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

def run_sample(sample):
    command = [
        conda_path, "run", "--no-capture-output",
        "-n", "genomad",
        "genomad", "end-to-end",
        "--cleanup",
        "--conservative",
        "--lenient-taxonomy", 
        "--threads", str(cores_per_sample),
        str(input_dir / sample / "final.contigs.renamed.fa"),
        str(output_dir/ sample),
        str(database)
    ]

    print(f"Starting: {sample}", flush=True)

    log_file = log_dir / f"{sample}.{tool_name}.log"

    with open(log_file, "w") as log:
        result = subprocess.run(
            command, stdout=log, stderr=subprocess.STDOUT
        )

    status = "Finished" if result.returncode == 0 else f"FAILED — check {log_file}"
    print(f"{sample}: {status}", flush=True)

with ThreadPoolExecutor(max_workers=max_samples) as executor:
    list(executor.map(run_sample, samples))

In [ ]:
# Add sample name in each viral contig
from pathlib import Path

input_dir = work_dir / "megahit"

for sample in samples:
    input_file = input_dir / sample / "final.contigs.fa"
    output_file = input_dir / sample / "final.contigs.renamed.fa"

    with open(input_file, "r") as infile, open(output_file, "w") as outfile:
        for line in infile:
            if line.startswith(">"):
                outfile.write(f">{sample}_{line[1:]}")
            else:
                outfile.write(line)

    print(f"Finished: {sample}")

In [ ]:
# Join all viral contigs

genomad_dir = work_dir / "genomad"
output_file = genomad_dir / "all_samples_virus.fna"

fasta_files = sorted(
    genomad_dir.glob(
        "*/final.contigs.renamed_summary/final.contigs.renamed_virus.fna"
    )
)

with open(output_file, "w") as outfile:
    for fasta_file in fasta_files:
        with open(fasta_file, "r") as infile:
            for line in infile:
                outfile.write(line)

        print(f"Added: {fasta_file}")

print(f"\nCombined {len(fasta_files)} files")
print(f"Output: {output_file}")

# Viral contigs quality

In [ ]:
import subprocess
from pathlib import Path

tool_name = "checkv"
threads = 18

input_file = Path("bled/genomad/all_samples_virus.fna")
output_dir = Path("bled/checkv")
log_dir = Path("bled/log") / tool_name
database = Path("DB/checkv-db-v1.5/")

output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

log_file = log_dir / f"{tool_name}.log"

command = [
    conda_path, "run", "--no-capture-output",
    "-n", "checkv",
    "checkv", "end_to_end",
    str(input_file),
    str(output_dir),
    "-t", str(threads),
    "-d", str(database)
]


print("Starting CheckV", flush=True)

with open(log_file, "w") as log:
    result = subprocess.run(
        command,
        stdout=log,
        stderr=subprocess.STDOUT
    )

if result.returncode == 0:
    print("CheckV finished successfully", flush=True)
else:
    print(f"CheckV FAILED — check {log_file}", flush=True)

In [13]:
# Remove extra _1 in header added by checkV
!sed -E '/^>/ s/_[0-9]+([[:space:]]|$)/\1/' bled/checkv/proviruses.fna > bled/checkv/proviruses_clean.fasta

In [14]:
# Join the virus and provirus files
!cat bled/checkv/viruses.fna bled/checkv/proviruses_clean.fasta > bled/checkv/all_checkv.fasta

In [16]:
# Select the Medium to complete viral contigs
import subprocess
from pathlib import Path

tool_name = "seqkit"
threads = 6

input_file = Path("bled/checkv/all_checkv.fasta")
id_file = Path("bled/hq_id.txt")
output_dir = Path("bled")
log_dir = Path("bled/log") / tool_name

output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "hq_contigs.fna"
log_file = log_dir / f"{tool_name}.log"


command = [
    conda_path, "run", "--no-capture-output",
    "-n", "seqkit",
    "seqkit", "grep",
    "--threads", str(threads),
    "--pattern-file", str(id_file),
    "--out-file", str(output_file),
    str(input_file),
]

print("Starting SeqKit grep", flush=True)

with open(log_file, "w") as log:
    result = subprocess.run(
        command,
        stdout=log,
        stderr=subprocess.STDOUT,
    )

if result.returncode == 0:
    print("SeqKit grep finished successfully", flush=True)
    print(f"Output: {output_file}")
else:
    print(f"SeqKit grep FAILED — check {log_file}", flush=True)

Starting SeqKit grep
SeqKit grep finished successfully
Output: bled/hq_contigs.fna


# Dereplication

In [18]:
from pathlib import Path
import subprocess
import shlex

# Settings
tool_name = "vclust"
conda_env = "vclust"
threads = 12

input_file = Path("bled/hq_contigs.fna")
output_dir = Path("bled/vclust")
log_dir = Path("bled/log") / tool_name

output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

# Output files
filter_file = output_dir / "fltr.txt"
ani_file = output_dir / "ani.tsv"
ids_file = output_dir / "ani.ids.tsv"
clusters_file = output_dir / "clusters.tsv"
log_file = log_dir / "all_samples.vclust.log"

conda_path = "/DKHF/users/johansaenz/miniforge3/bin/conda"


def run_command(step_name, arguments):
    command = [
        conda_path, "run", "--no-capture-output",
        "-n", conda_env,
        "vclust",
        *arguments,
    ]

    print(f"Starting: {step_name}", flush=True)

    with open(log_file, "a") as log:
        log.write(f"\n### {step_name}\n")
        log.write(shlex.join(command) + "\n\n")
        log.flush()

        result = subprocess.run(
            command,
            stdout=log,
            stderr=subprocess.STDOUT,
        )

    if result.returncode != 0:
        raise RuntimeError(
            f"{step_name} failed. Check: {log_file}"
        )

    print(f"Finished: {step_name}", flush=True)


# Check the input before starting
if not input_file.is_file():
    raise FileNotFoundError(f"Input file not found: {input_file}")


# Step 1: pre-alignment filter
run_command(
    "prefilter",
    [
        "prefilter",
        "-i", str(input_file),
        "-o", str(filter_file),
        "--min-ident", "0.95",
        "--threads", str(threads),
    ],
)


# Step 2: calculate ANI
run_command(
    "align",
    [
        "align",
        "-i", str(input_file),
        "-o", str(ani_file),
        "--filter", str(filter_file),
        "--out-ani", "0.95",
        "--out-qcov", "0.85",
        "--threads", str(threads),
    ],
)


# Step 3: cluster sequences
run_command(
    "cluster",
    [
        "cluster",
        "-i", str(ani_file),
        "-o", str(clusters_file),
        "--ids", str(ids_file),
        "--algorithm", "cd-hit",
        "--metric", "ani",
        "--ani", "0.95",
        "--qcov", "0.85",
        "--out-repr",
    ],
)

print("vclust analysis finished")
print(f"Clusters: {clusters_file}")
print(f"Log: {log_file}")

Starting: prefilter
Finished: prefilter
Starting: align
Finished: align
Starting: cluster
Finished: cluster
vclust analysis finished
Clusters: bled/vclust/clusters.tsv
Log: bled/log/vclust/all_samples.vclust.log


In [3]:
import subprocess
from pathlib import Path

tool_name = "seqkit"
threads = 6

input_file = Path("bled/hq_contigs.fna")
id_file = Path("bled/representative_id.txt")
output_dir = Path("bled")
log_dir = Path("bled/log") / tool_name

output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "representative_votus.fna"
log_file = log_dir / f"{tool_name}.log"

conda_path = "/DKHF/users/johansaenz/miniforge3/bin/conda"

command = [
    conda_path, "run", "--no-capture-output",
    "-n", "seqkit",
    "seqkit", "grep",
    "--threads", str(threads),
    "--pattern-file", str(id_file),
    "--out-file", str(output_file),
    str(input_file),
]

print("Starting SeqKit grep", flush=True)

with open(log_file, "w") as log:
    result = subprocess.run(
        command,
        stdout=log,
        stderr=subprocess.STDOUT,
    )

if result.returncode == 0:
    print("SeqKit grep finished successfully", flush=True)
    print(f"Output: {output_file}")
else:
    print(f"SeqKit grep FAILED — check {log_file}", flush=True)

Starting SeqKit grep
SeqKit grep finished successfully
Output: bled/representative_votus.fna


# vOTUs abundance

In [3]:
import subprocess
from concurrent.futures import ThreadPoolExecutor

max_samples = 2
cores_per_sample = 14
tool_name = "coverm"

input_dir = Path("bled/cutadapt/")
output_dir = Path("bled/coverm_all")
log_dir = Path("bled/log") / tool_name
viral_otu = Path("/scratch/johansaenz/bled/genomad/all_samples_virus.fna")

output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

conda_path = "/DKHF/users/johansaenz/miniforge3/bin/conda"


def run_sample(sample):
    command = [
        conda_path, "run", "--no-capture-output",
        "-n", "coverm",
        "TMPDIR=.", "coverm", "contig",
        "--methods", "count",
        "--threads", str(cores_per_sample),
        "-c", 
        str(input_dir / f"{sample}_1_cut.fq.gz"),
        str(input_dir / f"{sample}_2_cut.fq.gz"),
        "-r", str(viral_otu),
        "-o", str(output_dir / f"{sample}.coverm.tsv"),
        "--min-read-percent-identity", "90", 
        "--min-read-aligned-percent", "75",
        "--exclude-supplementary",
        #"--min-covered-fraction", "75",
    ]

    print(f"Starting: {sample}", flush=True)

    log_file = log_dir / f"{sample}.{tool_name}.log"

    with open(log_file, "w") as log:
        result = subprocess.run(
            command, stdout=log, stderr=subprocess.STDOUT
        )

    status = "Finished" if result.returncode == 0 else f"FAILED — check {log_file}"
    print(f"{sample}: {status}", flush=True)

with ThreadPoolExecutor(max_workers=max_samples) as executor:
    list(executor.map(run_sample, samples))

Starting: W000_E_RP
Starting: W000_T2_RP
W000_T2_RP: Finished
Starting: W001_T2_RP
W000_E_RP: Finished
Starting: W002_T2_RP
W002_T2_RP: Finished
W001_T2_RP: Finished


# AMR genes search

In [9]:
import os
import subprocess
from pathlib import Path

# Configuration
cores = 14
conda_path = "/DKHF/users/johansaenz/miniforge3/bin/conda"

work_dir = Path("/scratch/johansaenz/bled/checkv/")
viral_otu = work_dir / "all_checkv.fasta"

# CHANGE this to the actual absolute path
card_json = Path("/scratch/johansaenz/DB/card.json")

output_dir = work_dir / "rgi"
log_dir = work_dir / "log/rgi"

# Use the Agg Matplotlib backend for RGI subprocesses
rgi_env = os.environ.copy()
rgi_env["MPLBACKEND"] = "Agg"

# Check input files
for path in (Path(conda_path), viral_otu, card_json):
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

rgi_command = [
    conda_path, "run", "--no-capture-output",
    "-n", "rgi",
    "rgi",
]


def run_step(arguments, log_file):
    with log_file.open("w") as log:
        try:
            subprocess.run(
                rgi_command + arguments,
                cwd=work_dir,
                env=rgi_env,
                stdout=log,
                stderr=subprocess.STDOUT,
                check=True,
            )
        except subprocess.CalledProcessError as error:
            raise RuntimeError(
                f"RGI failed with exit code {error.returncode}. "
                f"Check log: {log_file}"
            ) from error


# 1. Load CARD into the local working directory
print("Loading CARD database...", flush=True)

run_step(
    [
        "load",
        "--card_json", str(card_json),
        "--local",
    ],
    log_dir / "card_load.log",
)

# 2. Analyze representative vOTUs
print("Running RGI on representative vOTUs...", flush=True)

run_step(
    [
        "main",
        "--threads", str(cores),
        "--input_sequence", str(viral_otu),
        "--input_type", "contig",
        "--output_file", str(output_dir / "rgi_out"),
        #"--include_loose",
        "--local",
        "--clean",
    ],
    log_dir / "representative_votus.rgi.log",
)

print(f"Finished. Results directory: {output_dir}", flush=True)

Loading CARD database...
Running RGI on representative vOTUs...
Finished. Results directory: /scratch/johansaenz/bled/checkv/rgi


# Export conda enviroments

In [ ]:
mkdir -p reproducibility

# Export the enviroment
conda env export -n genomad > reproducibility/genomad.yml